# 실습 - 시각 질의응답

**의료 영상에 질문하여 답을 얻음**

**작업 개요**

- 작업 이름 — `visual-question-answering`
- 입력 — **이미지 + 질문** (문서 질의응답과 동일 구조)
- 출력 — 답변
- 문서가 아닌 **일반 이미지** 대상

**데이터**

- 뇌종양 MRI 이미지
- `Question` · `Answer` 열에 질문과 정답이 함께 저장
- 정답은 **서술 형태** — 위치 · 종류까지 기술

**두 모델을 비교**

| 모델 | 학습 데이터 | 방식 |
|---|---|---|
| ViLT | 일상 사진 | 후보 중 선택 |
| BLIP-LoRA-VQARAD | **방사선 영상** | 문장 생성 |

- 의료 모델은 **LoRA**로 미세조정된 것
- 오후 실습에서 다룰 기법 — 이미 만들어진 결과물을 먼저 사용

**실습 순서**

- 환경 준비
- 데이터 불러오기 · 구조 확인
- 일반 모델(ViLT) 실행
- 의료 특화 모델 불러오기 · 실행
- 두 결과와 정답 비교

---
## 0. 환경 준비

- `datasets` — Hub 데이터셋 내려받기
- `transformers` — 사전학습 모델로 추론 수행
- `huggingface_hub` — 저장소 파일 목록 확인

In [ ]:
from datasets import load_dataset
from transformers import pipeline
from huggingface_hub import list_repo_files
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("준비 완료")
print("장치:", device)

---
## 1. 데이터 준비

### 1-1. 데이터셋 불러오기

- 뇌종양 MRI 이미지와 질문·답변 쌍
- `split="train[:50]"` — 앞 50장만 사용

In [ ]:
ds = load_dataset(
    "Kaith-jeet123/brain_tumor_vqa",
    split="train[:50]"
)

print("열:", ds.column_names)
ds

### 1-2. 항목 확인

- 열 이름이 **대문자로 시작** — `Question` · `Answer`
- 정답이 서술 형태 — 위치와 종류까지 기술

In [ ]:
item = ds[0]

print("질문:", item["Question"])
print()
print("정답:", item["Answer"])
print()

image = item["image"].convert("RGB")
print("이미지:", image.mode, image.size)
image

---
## 2. 일반 모델 — ViLT

- 일상 사진으로 학습된 시각 질의응답 모델
- `pipeline` 으로 바로 사용 가능
- 출력 — 답변 후보와 점수, **미리 정해진 후보 중 선택**

In [ ]:
vilt = pipeline(
    task="visual-question-answering",
    model="dandelin/vilt-b32-finetuned-vqa",
    device=0 if device == "cuda" else -1
)

for r in vilt(image=image, question=item["Question"])[:3]:
    print(r["answer"], "-", round(r["score"], 3))

**확인 사항**

- 답변이 **단어 하나** — `yes` · `no` 등
- 데이터의 정답(서술문)과 형태가 근본적으로 다름

---
## 3. 의료 특화 모델

### 3-1. 저장소 구조 확인

- LoRA 모델은 두 가지 형태로 배포됨
- **병합본** — `model.safetensors` 등, 바로 사용 가능
- **어댑터만** — `adapter_model.safetensors`, 원본 모델과 결합 필요

어떤 형태인지 먼저 확인한다.

In [ ]:
MED_CKPT = "dineshcr7/Final-BLIP-LORA_VQA_RAD"

files = list_repo_files(MED_CKPT)
for f in sorted(files):
    print(f)

is_adapter = any("adapter" in f for f in files)
print()
print("형태:", "어댑터만" if is_adapter else "병합본")

### 3-2. 모델 불러오기

- 병합본이면 그대로 불러옴
- 어댑터만 있으면 원본 BLIP과 결합 (`peft` 필요)
- 두 경우를 모두 처리

In [ ]:
from transformers import BlipProcessor, BlipForQuestionAnswering

BASE_CKPT = "Salesforce/blip-vqa-base"

if is_adapter:
    from peft import PeftModel
    base = BlipForQuestionAnswering.from_pretrained(BASE_CKPT)
    med_model = PeftModel.from_pretrained(base, MED_CKPT).to(device)
    med_proc = BlipProcessor.from_pretrained(BASE_CKPT)
    print("어댑터 결합 완료")
else:
    med_model = BlipForQuestionAnswering.from_pretrained(MED_CKPT).to(device)
    try:
        med_proc = BlipProcessor.from_pretrained(MED_CKPT)
    except Exception:
        med_proc = BlipProcessor.from_pretrained(BASE_CKPT)
    print("병합본 불러오기 완료")

med_model.eval()
print("준비 완료")

### 3-3. 질문 함수 정의

- 이미지와 질문을 함께 처리기에 입력
- `generate` — 답변을 문장으로 생성
- 앞선 모델들과 달리 **후보 목록이 없음**

In [ ]:
def ask_med(img, question, max_new_tokens=32):
    """의료 특화 모델에 질문"""
    img = img.convert("RGB")
    inputs = med_proc(img, question, return_tensors="pt").to(device)

    with torch.no_grad():
        out = med_model.generate(**inputs, max_new_tokens=max_new_tokens)

    return med_proc.decode(out[0], skip_special_tokens=True).strip()


print("함수 준비 완료")

### 3-4. 질문 실행

In [ ]:
answer = ask_med(image, item["Question"])

print("질문:", item["Question"])
print()
print("모델 답변:", answer)
print()
print("데이터 정답:", item["Answer"])

---
## 4. 두 모델 비교

- 같은 이미지 · 같은 질문에 두 모델을 적용
- 학습 데이터의 차이가 결과에 어떻게 나타나는지 확인

In [ ]:
q = item["Question"]

print("질문:", q)
print()
print("[ViLT — 일상 사진 학습]")
print("  ", vilt(image=image, question=q)[0]["answer"])
print()
print("[BLIP-LoRA — 방사선 영상 학습]")
print("  ", ask_med(image, q))
print()
print("[데이터 정답]")
print("  ", item["Answer"])

image

### 4-1. 여러 사례 비교

- 앞부분 여러 항목에 대해 두 모델의 답변을 나란히 확인

In [ ]:
for i in range(3):
    img = ds[i]["image"].convert("RGB")
    q = ds[i]["Question"]
    a = ds[i]["Answer"]

    print("=" * 60)
    print("질문:", q[:100])
    print("  ViLT      :", vilt(image=img, question=q)[0]["answer"])
    print("  BLIP-LoRA :", ask_med(img, q))
    print("  정답       :", a[:150])
    print()

**확인 사항**

- **ViLT** — 단어 하나, 근거를 알 수 없음
- **BLIP-LoRA** — 방사선 영상으로 학습, 의료 용어 사용
- **데이터 정답** — 위치 · 종류까지 상세 기술

학습 데이터의 성격이 답변 수준을 결정한다. 다만 특화 모델도 데이터 정답만큼 상세하지는 않을 수 있다.

---
## 5. 질문을 바꾸어 실행

- 질문 표현에 따라 답변이 달라짐
- 단순 질문과 전문 질문의 결과 비교

In [ ]:
questions = [
    "Is there a tumor?",
    "Where is the lesion located?",
    "What imaging modality is this?",
    "Is this a CT or MRI?",
]

for q in questions:
    print(q)
    print("  ViLT      :", vilt(image=image, question=q)[0]["answer"])
    print("  BLIP-LoRA :", ask_med(image, q))
    print()

---
## 6. 다른 사례 확인

- 이미지 번호를 바꾸어 다른 사례 확인
- 질문 표현을 바꾸어 결과 변화 관찰
- 본인 분야 이미지에 적용 — 어떤 질문에 답할 수 있는지 확인

In [ ]:
n = 5
img = ds[n]["image"].convert("RGB")

print("데이터 질문:", ds[n]["Question"])
print()
print("BLIP-LoRA:", ask_med(img, ds[n]["Question"]))
print()
print("데이터 정답:", ds[n]["Answer"][:200])

img

---
## 7. 정리

**수행 내용**

- 같은 이미지 · 같은 질문에 두 모델 적용
- 일반 모델(ViLT)과 의료 특화 모델(BLIP-LoRA) 비교
- 데이터의 정답과 대조

**확인한 사항**

- 학습 데이터의 성격이 **답변 수준을 결정**
- 일반 모델 — 후보 중 단어 하나 선택
- 특화 모델 — 문장 생성, 해당 분야 용어 사용
- 특화 모델도 전문가 수준의 상세 서술에는 미치지 못함

**LoRA와의 연결**

- 이 의료 모델은 **LoRA**로 미세조정된 결과물
- 원본 모델은 그대로 두고 소수 파라미터만 학습하는 방식
- 오후 실습에서 동일한 기법을 직접 수행

**pipeline의 범위**

- pipeline은 편리하나 **모든 모델을 지원하지는 않음**
- 지원되지 않는 모델은 처리기와 모델을 **직접 호출**
- 모델 카드의 사용 예제를 확인하는 습관 필요

**주의**

- **의료 판단에 사용 불가** — 교육 목적의 예시
- 생성 방식 — 실행할 때마다 표현이 달라질 수 있음

---

데이터 출처: Kaith-jeet123/brain_tumor_vqa
모델 출처: dineshcr7/Final-BLIP-LORA_VQA_RAD (VQA-RAD LoRA 미세조정)